# LoRA + SFT on DGX Spark: Can We Fine-Tune Granite on a Desktop Supercomputer?

**Algorithm:** LoRA (Low-Rank Adaptation) with Supervised Fine-Tuning  
**Model:** `ibm-granite/granite-3.3-2b-instruct`  
**Hardware:** NVIDIA DGX Spark (GB10, 130.7 GB unified memory)  
**Dataset:** `b-mc2/sql-create-context` (natural language → SQL)  

---

This notebook documents our attempt to run LoRA+SFT fine-tuning on a DGX Spark using
[Training Hub](https://github.com/instructlab/training-hub). The DGX Spark is NVIDIA's
"desktop AI supercomputer" — a Grace Blackwell system with 130.7 GB of shared CPU/GPU memory
and an ARM (aarch64) processor. It's a fascinating piece of hardware, but it's also bleeding-edge:
compute capability 12.1, CUDA 13.0, and an ARM CPU mean that many ML libraries don't have
pre-built wheels yet.

We're not just showing the happy path — we're documenting what worked, what broke, and what
workarounds were needed. That's the real story.

## 1. DGX Spark System Profile

Let's see exactly what hardware and software we're working with.

> **Estimated run time:** ~5 seconds

In [ ]:
import platform
import subprocess
import torch

print("=" * 60)
print("DGX Spark System Profile")
print("=" * 60)

# CPU & OS
print(f"\nPlatform:    {platform.machine()}")
print(f"Processor:   {platform.processor() or 'ARM (Grace)'}")
print(f"OS:          {platform.platform()}")

# GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    cc = torch.cuda.get_device_capability(0)
    print(f"\nGPU:         {gpu_name}")
    print(f"GPU Memory:  {gpu_mem:.1f} GB")
    print(f"Compute Cap: {cc[0]}.{cc[1]}")
    print(f"CUDA:        {torch.version.cuda}")
else:
    print("\nNo CUDA GPU detected!")

# PyTorch
print(f"\nPyTorch:     {torch.__version__}")
print(f"BF16:        {torch.cuda.is_bf16_supported()}")

# Shared memory (DGX Spark uses unified memory)
import os
total_ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
print(f"\nSystem RAM:  {total_ram:.1f} GB (shared with GPU on DGX Spark)")

## 2. Environment Check — What's Installed, What's Missing

Training Hub's LoRA backend defaults to **Unsloth** for optimized LoRA training. Let's check
what we have and what we're missing.

> **Estimated run time:** ~2 seconds

In [ ]:
# Check installed packages relevant to LoRA training
packages = [
    "training-hub", "instructlab-training", "mini-trainer",
    "peft", "bitsandbytes", "trl", "accelerate",
    "transformers", "datasets", "torch",
    "unsloth", "xformers", "flash-attn", "deepspeed",
    "liger-kernel", "mamba-ssm"
]

import importlib.metadata

print(f"{'Package':<25} {'Status':<15} {'Version'}")
print("-" * 55)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:<25} {'INSTALLED':<15} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:<25} {'MISSING':<15} —")

### 2a. Attempting to Install Unsloth

Unsloth is the default backend for `lora_sft()`. Let's try installing it.

> **Expectation:** This will likely fail on DGX Spark. Unsloth has custom CUDA kernels that
> may not have pre-built wheels for ARM aarch64 + compute capability 12.1. We document the
> failure and plan a fallback.

> **Estimated run time:** 30–120 seconds (pip resolve + possible build attempt)

In [ ]:
%%time
# Attempt to install unsloth
# This cell documents what happens — success or failure is equally valuable
!pip install unsloth 2>&1 | tail -20

In [ ]:
# Check if unsloth is now importable
try:
    import unsloth
    print(f"Unsloth installed successfully! Version: {unsloth.__version__}")
    UNSLOTH_AVAILABLE = True
except ImportError as e:
    print(f"Unsloth import failed: {e}")
    print("\nWe'll proceed with a PEFT/TRL fallback approach.")
    UNSLOTH_AVAILABLE = False

In [ ]:
%%time
# Also try xformers (used by some LoRA backends for memory-efficient attention)
!pip install xformers 2>&1 | tail -10

try:
    import xformers
    print(f"\nxformers installed! Version: {xformers.__version__}")
except ImportError as e:
    print(f"\nxformers not available: {e}")
    print("Not critical — PyTorch's built-in SDPA attention will be used instead.")

## 3. Memory Estimation — Will It Fit?

Before we try training, let's use Training Hub's memory estimator to check if our
configurations will fit in 130.7 GB of shared memory.

> **Estimated run time:** ~30–60 seconds (downloads model config from HuggingFace on first run)

In [ ]:
%%time
from training_hub import estimate

GPU_MEM_BYTES = int(130.7 * 1024**3)  # 130.7 GB in bytes
MODEL = "ibm-granite/granite-3.3-2b-instruct"

print("=" * 60)
print("Memory Estimate: QLoRA (4-bit) — Granite 2B")
print("=" * 60)
qlora_est = estimate(
    training_method="qlora",
    num_gpus=1,
    gpu_memory=GPU_MEM_BYTES,
    model_path=MODEL,
    verbose=2,
)

In [ ]:
%%time
print("=" * 60)
print("Memory Estimate: LoRA (full precision) — Granite 2B")
print("=" * 60)
lora_est = estimate(
    training_method="lora",
    num_gpus=1,
    gpu_memory=GPU_MEM_BYTES,
    model_path=MODEL,
    verbose=2,
)

In [ ]:
# Summary
def fmt_gb(b):
    return f"{b / 1024**3:.1f} GB"

print("\nMemory Summary:")
print(f"  Available:       {fmt_gb(GPU_MEM_BYTES)}")
print(f"  QLoRA estimate:  {fmt_gb(qlora_est[0])} – {fmt_gb(qlora_est[2])}")
print(f"  LoRA estimate:   {fmt_gb(lora_est[0])} – {fmt_gb(lora_est[2])}")
print(f"\nVerdict: Both should fit comfortably. QLoRA is the safest bet.")

## 4. Dataset Preparation

We're using the **sql-create-context** dataset — it contains natural language questions paired
with SQL queries and table schemas. Small, public, and easy to understand.

Training Hub expects JSONL with a `messages` field containing chat-format conversations.

> **Estimated run time:** ~10–30 seconds (dataset download from HuggingFace)

In [ ]:
%%time
from datasets import load_dataset
import json
import os

# Download the dataset
ds = load_dataset("b-mc2/sql-create-context", split="train")
print(f"Dataset size: {len(ds)} examples")
print(f"Columns: {ds.column_names}")
print(f"\nSample entry:")
print(json.dumps(ds[0], indent=2))

In [ ]:
# Convert to Training Hub's messages JSONL format
# We'll use a subset (2000 examples) to keep training time reasonable

SUBSET_SIZE = 2000
DATA_PATH = "data/sql_train.jsonl"

os.makedirs("data", exist_ok=True)

system_prompt = (
    "You are a SQL expert. Given a natural language question and the relevant table schema, "
    "write the correct SQL query."
)

with open(DATA_PATH, "w") as f:
    for i, row in enumerate(ds):
        if i >= SUBSET_SIZE:
            break
        user_content = f"Question: {row['question']}\n\nContext:\n{row['context']}"
        record = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": row["answer"]},
            ]
        }
        f.write(json.dumps(record) + "\n")

print(f"Wrote {min(SUBSET_SIZE, len(ds))} examples to {DATA_PATH}")
print(f"File size: {os.path.getsize(DATA_PATH) / 1024:.1f} KB")

# Preview first record
with open(DATA_PATH) as f:
    sample = json.loads(f.readline())
print(f"\nPreview:")
print(json.dumps(sample, indent=2))

## 5. Training Run — LoRA+SFT with Training Hub

Now for the main event. We'll try Training Hub's `lora_sft()` function.

**Strategy:**
1. If Unsloth is available → use it (default backend)
2. If Unsloth failed → try with `backend="peft"` or manual PEFT/TRL approach

> **Estimated run time:** 5–15 minutes for QLoRA on Granite 2B with 2000 examples
> (includes model download on first run: ~2 min, plus ~3–10 min training)

In [ ]:
import time
import os

CKPT_DIR = "checkpoints/lora_sft_granite2b"
os.makedirs(CKPT_DIR, exist_ok=True)

print("Starting LoRA+SFT training...")
print(f"  Model:    {MODEL}")
print(f"  Data:     {DATA_PATH}")
print(f"  Output:   {CKPT_DIR}")
print(f"  Backend:  {'unsloth' if UNSLOTH_AVAILABLE else 'unsloth (may fallback)'}")
print()

In [ ]:
from training_hub import lora_sft

start_time = time.time()

try:
    result = lora_sft(
        model_path=MODEL,
        data_path=os.path.abspath(DATA_PATH),
        ckpt_output_dir=os.path.abspath(CKPT_DIR),
        # QLoRA 4-bit quantization for minimum memory
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="bfloat16",
        bnb_4bit_use_double_quant=True,
        # LoRA config
        lora_r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        # Training config
        num_epochs=1,
        effective_batch_size=8,
        max_seq_len=512,
        learning_rate=2e-4,
        lr_scheduler="cosine",
        warmup_steps=10,
        bf16=True,
        # Single GPU
        nproc_per_node=1,
        # Logging
        logging_steps=25,
        save_steps=500,
    )
    elapsed = time.time() - start_time
    print(f"\nTraining completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

except Exception as e:
    elapsed = time.time() - start_time
    print(f"\nTraining failed after {elapsed:.1f}s")
    print(f"Error: {type(e).__name__}: {e}")
    print("\nSee the Troubleshooting section below for analysis and workarounds.")
    import traceback
    traceback.print_exc()

## 6. Fallback — Manual PEFT/TRL Approach

If Training Hub's `lora_sft()` failed (likely due to Unsloth unavailability), we can try
a manual LoRA fine-tuning approach using PEFT and TRL directly. These libraries are
pure Python with no custom CUDA kernels, so they should work on any platform.

> **Run this section only if the Training Hub call above failed.**

> **Estimated run time:** 5–15 minutes (model loading ~1–2 min, training ~3–10 min)

In [ ]:
# Manual PEFT/TRL fallback
# Uncomment and run this cell if lora_sft() failed above

MANUAL_FALLBACK = False  # Set to True if needed

if MANUAL_FALLBACK:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig
    from datasets import load_dataset
    import torch

    # 4-bit quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print("Loading model with 4-bit quantization...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model)

    # LoRA config
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Load our JSONL dataset
    train_ds = load_dataset("json", data_files=DATA_PATH, split="train")

    # Format for TRL's SFTTrainer
    def format_messages(example):
        return {"text": tokenizer.apply_chat_template(
            example["messages"], tokenize=False, add_generation_prompt=False
        )}

    train_ds = train_ds.map(format_messages)

    # Training config
    training_args = SFTConfig(
        output_dir=CKPT_DIR + "_manual",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        max_seq_length=512,
        bf16=True,
        logging_steps=25,
        save_steps=500,
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        processing_class=tokenizer,
    )

    print("\nStarting manual PEFT/TRL training...")
    start_time = time.time()
    trainer.train()
    elapsed = time.time() - start_time
    print(f"\nManual training completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

    # Save
    trainer.save_model(CKPT_DIR + "_manual/final")
    print(f"Model saved to {CKPT_DIR}_manual/final")
else:
    print("Manual fallback not needed — Training Hub succeeded (or set MANUAL_FALLBACK=True to try).")

## 7. GPU Memory Usage

Let's check how much memory training actually used.

In [ ]:
if torch.cuda.is_available():
    allocated = torch.cuda.max_memory_allocated() / 1024**3
    reserved = torch.cuda.max_memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_mem / 1024**3

    print("Peak GPU Memory Usage:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")
    print(f"  Total:     {total:.1f} GB")
    print(f"  Headroom:  {total - reserved:.1f} GB")
else:
    print("No CUDA device available for memory stats.")

## 8. Bonus: QLoRA on Granite 8B

The DGX Spark has enough memory that we might be able to QLoRA fine-tune the 8B model too.
The estimator says 7.8–10.6 GB — well within our 130.7 GB budget.

> **This is a stretch goal.** Skip if time is limited.

> **Estimated run time:** 15–45 minutes for 8B QLoRA (model download ~5 min, training ~10–40 min)

In [ ]:
%%time
# Memory estimate for 8B QLoRA
MODEL_8B = "ibm-granite/granite-3.3-8b-instruct"

print("Memory Estimate: QLoRA (4-bit) — Granite 8B")
print("=" * 60)
qlora_8b_est = estimate(
    training_method="qlora",
    num_gpus=1,
    gpu_memory=GPU_MEM_BYTES,
    model_path=MODEL_8B,
    verbose=2,
)
print(f"\nEstimate: {fmt_gb(qlora_8b_est[0])} – {fmt_gb(qlora_8b_est[2])}")
print(f"Available: {fmt_gb(GPU_MEM_BYTES)}")
print("Verdict: Should fit!")

In [ ]:
# Uncomment to run 8B QLoRA training
# Warning: will take significantly longer than 2B

RUN_8B = False  # Set to True to attempt 8B training

if RUN_8B:
    CKPT_DIR_8B = "checkpoints/lora_sft_granite8b"
    os.makedirs(CKPT_DIR_8B, exist_ok=True)

    # Clear GPU memory from previous run
    torch.cuda.empty_cache()
    import gc; gc.collect()

    print("Starting QLoRA training on Granite 8B...")
    start_time = time.time()

    try:
        result_8b = lora_sft(
            model_path=MODEL_8B,
            data_path=os.path.abspath(DATA_PATH),
            ckpt_output_dir=os.path.abspath(CKPT_DIR_8B),
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype="bfloat16",
            bnb_4bit_use_double_quant=True,
            lora_r=16,
            lora_alpha=32,
            lora_dropout=0.0,
            num_epochs=1,
            effective_batch_size=4,  # Smaller batch for 8B
            max_seq_len=512,
            learning_rate=1e-4,
            lr_scheduler="cosine",
            warmup_steps=10,
            bf16=True,
            nproc_per_node=1,
            logging_steps=25,
            save_steps=500,
        )
        elapsed = time.time() - start_time
        print(f"\n8B training completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"\n8B training failed after {elapsed:.1f}s")
        print(f"Error: {e}")
else:
    print("8B training skipped. Set RUN_8B=True to attempt.")

## 9. Troubleshooting Log

This section documents issues encountered and workarounds attempted.

### Known Issues on DGX Spark

| Issue | Detail | Workaround |
|-------|--------|------------|
| Compute capability 12.1 | PyTorch warns max supported is 12.0 | Usually works anyway — warning only |
| Unsloth wheels | No pre-built wheels for ARM aarch64 | Manual PEFT/TRL fallback (Section 6) |
| xformers | No ARM wheels available | Use PyTorch's built-in SDPA attention |
| CUDA arch | sm_121 not in default TORCH_CUDA_ARCH_LIST | May need `TORCH_CUDA_ARCH_LIST="12.1"` for source builds |

### What Worked
- *Fill in after running*

### What Didn't Work
- *Fill in after running*

### Key Observations
- *Fill in after running*

## 10. Results & Next Steps

### Summary
- **Algorithm:** LoRA + SFT (QLoRA 4-bit)
- **Model:** Granite 2B (primary), Granite 8B (stretch)
- **Hardware:** DGX Spark (GB10, 130.7 GB unified memory)
- **Training time:** *Fill in after running*
- **Peak memory:** *Fill in after running*
- **Status:** *Fill in after running*

### Recommendations
- LoRA/QLoRA is the most accessible training method on DGX Spark
- The massive shared memory means even 8B models can be QLoRA'd comfortably
- ARM + Blackwell ecosystem is still maturing — expect dependency challenges

### Next
- [Notebook 2: Full SFT →](02_sft_spark.ipynb)
- [Notebook 3: OSFT →](03_osft_spark.ipynb)